# 4. Production Pipelines with Imbalanced Data (`imblearn.pipeline.Pipeline`)

This notebook covers:
1. **The Critical Trap of `sklearn.pipeline.Pipeline` with Resampling**: Why Scikit-Learn pipelines cannot execute SMOTE and fail during cross-validation.
2. **The Correct Solution (`imblearn.pipeline.Pipeline`)**: Automatically applying SMOTE strictly during `.fit()` on training folds while leaving evaluation/test data un-resampled.
3. **Unified Preprocessing + Synthetic Oversampling + Classifier**:
   - `ColumnTransformer` (Scaling + Encoding)
   - `SMOTE` (Synthetic Generation on encoded numeric vectors)
   - `RandomForestClassifier`
4. **Proper Skewed Metrics**: Moving beyond standard ROC-AUC to Precision-Recall AUC (`PR-AUC` / Average Precision).

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, PrecisionRecallDisplay, average_precision_score

# Critical Import: Use imblearn's Pipeline, NOT sklearn's Pipeline
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

# 1. Create realistic tabular dataset with mixed features and extreme 95:5 imbalance
X_raw, y_raw = make_classification(
    n_samples=1200,
    n_features=4,
    n_informative=3,
    n_redundant=1,
    weights=[0.95, 0.05],  # 95% Legit (0), 5% Fraud (1)
    random_state=42
)

# Convert to DataFrame with mixed data types (Numerical + Categorical)
df = pd.DataFrame(X_raw, columns=['Transaction_Amount', 'Account_Age_Days', 'Login_Attempts', 'Risk_Score'])
df['Device_Type'] = np.random.choice(['Mobile', 'Desktop', 'Tablet'], size=len(df), p=[0.6, 0.3, 0.1])
df['Is_Fraud'] = y_raw

print("=== 1. RAW DATASET OVERVIEW ===")
display(df.head())
print("\nClass Distribution:")
display(df['Is_Fraud'].value_counts().to_frame(name='Count').assign(
    Proportion=df['Is_Fraud'].value_counts(normalize=True).map('{:.1%}'.format)
))

=== 1. RAW DATASET OVERVIEW ===


,Transaction_Amount,Account_Age_Days,Login_Attempts,Risk_Score,Device_Type,Is_Fraud
0,-1.346072,0.138080,0.145693,0.971393,Mobile,0
1,0.046221,-2.411918,-2.107142,-0.327620,Mobile,1
2,-0.717352,-1.404037,-0.852919,-0.782067,Mobile,0
3,-1.009276,-1.353631,-0.741509,-0.761564,Desktop,0
4,-0.934501,-1.361743,-1.310873,0.928906,Mobile,0



Class Distribution:


,Count,Proportion
Is_Fraud,,
0,1136,94.7%
1,64,5.3%


---
## Step 1: Stratified Train / Test Split

Split the dataset into an 80/20 train/test split with stratification enabled.

In [ ]:
X = df.drop(columns=['Is_Fraud']).copy()
y = df['Is_Fraud'].copy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

print(f"Train samples: {len(X_train)} (Fraud: {y_train.sum()})")
print(f"Test samples:  {len(X_test)} (Fraud: {y_test.sum()})")

Train samples: 960 (Fraud: 51)
Test samples:  240 (Fraud: 13)


---
## Step 2: Build the Preprocessor (`ColumnTransformer`)

SMOTE relies on Euclidean distance ($k$-Nearest Neighbors), so:
* All categorical columns must be converted to numbers (`OneHotEncoder`).
* All numerical columns must be on the same scale (`StandardScaler`).

In [7]:
preprocessor = ColumnTransformer(transformers=[
    ('cat', OneHotEncoder(sparse_output=False, drop='first'), ['Device_Type']),
    ('num', StandardScaler(), ['Transaction_Amount', 'Account_Age_Days', 'Login_Attempts', 'Risk_Score'])
], remainder = 'passthrough')

---
## Step 3: Assemble the End-to-End Imbalanced Pipeline

> **Why `imblearn.pipeline.Pipeline`?**
> Standard `sklearn.pipeline.Pipeline` requires every intermediate step to implement `.transform()`.
> Resamplers like `SMOTE` implement `.fit_resample()`, not `.transform()`.
> `imblearn.pipeline.Pipeline` seamlessly applies SMOTE **only during `.fit()`** on training folds, and automatically **bypasses SMOTE during `.predict()` or test evaluation**.

In [8]:
pipeline = ImbPipeline(steps=[
    ('preprocessor', preprocessor),
    ('sampler', SMOTE(random_state=42, k_neighbors=5)),
    ('classifier', RandomForestClassifier(random_state=42, n_estimators=100))
])

pipeline.fit(X_train,y_train)
y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]

print("=== TEST SET PERFORMANCE (IMBLEARN PIPELINE) ===")
print(classification_report(y_test, y_pred, target_names=['Legit (0)', 'Fraud (1)']))

=== TEST SET PERFORMANCE (IMBLEARN PIPELINE) ===
              precision    recall  f1-score   support

   Legit (0)       0.98      0.98      0.98       227
   Fraud (1)       0.64      0.69      0.67        13

    accuracy                           0.96       240
   macro avg       0.81      0.84      0.82       240
weighted avg       0.96      0.96      0.96       240



---
## Step 4: Leak-Free Cross-Validation

Using `imblearn.pipeline.Pipeline` inside `cross_val_score` guarantees that SMOTE is executed **inside each training fold independently**, preventing synthetic samples from leaking into validation folds.